In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from torch_geometric.data import Data


# 1. Check that the shared model class and metric function are available
if (
    "GCNForecaster" not in globals()
    or "calculate_metrics" not in globals()
):
    raise RuntimeError(
        "Run Code 7 first to define GCNForecaster "
        "and calculate_metrics()."
    )


# 2. Define paths and permutation settings
graph_folder = Path(
    "results/step_06"
)

model_folder = Path(
    "results/step_07"
)

output_folder = Path(
    "results/step_11"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

PERMUTATION_SEED = 142
N_REPEATS = 10

device = torch.device("cpu")

rng = np.random.default_rng(
    PERMUTATION_SEED
)


# 3. Load the graph data and the trained GCN checkpoint
graph_payload = torch.load(
    graph_folder / "graph_datasets.pt",
    map_location="cpu",
    weights_only=True
)

checkpoint = torch.load(
    model_folder / "GCN_best_model.pt",
    map_location="cpu",
    weights_only=True
)

test_graphs = [
    Data(**graph)
    for graph in graph_payload["test"]
]

references = graph_payload["references"]
feature_names = graph_payload["feature_names"]

n_parts = int(
    graph_payload["n_parts"]
)

n_features = int(
    graph_payload["n_features"]
)

encoder_length = int(
    graph_payload["encoder_length"]
)

target_scale = float(
    checkpoint["target_scale"]
)

assert (
    checkpoint["references"]
    == references
)

assert (
    checkpoint["feature_names"]
    == feature_names
)

assert np.isclose(
    target_scale,
    graph_payload["target_scale"]
)


# 4. Restore the best GCN
gcn_model = GCNForecaster(
    input_channels=checkpoint["input_channels"],
    hidden_channels=checkpoint["hidden_channels"],
    prediction_horizon=checkpoint["prediction_horizon"]
).to(device)

gcn_model.load_state_dict(
    checkpoint["state_dict"]
)

gcn_model.eval()


# 5. Define prediction in original quantity units
def predict_graphs(model, graphs):

    actual_values = []
    predicted_values = []

    model.eval()

    with torch.no_grad():

        for graph in graphs:

            graph = graph.to(device)

            predictions = model(
                graph
            )

            predictions_raw = (
                predictions.cpu().numpy()
                * target_scale
            )

            predictions_raw = np.maximum(
                predictions_raw,
                0
            )

            actual_values.append(
                graph.y_raw.cpu().numpy()
            )

            predicted_values.append(
                predictions_raw
            )

    return (
        np.concatenate(actual_values, axis=0),
        np.concatenate(predicted_values, axis=0)
    )


# 6. Calculate the baseline test performance
baseline_actual, baseline_predictions = predict_graphs(
    gcn_model,
    test_graphs
)

baseline_metrics = calculate_metrics(
    baseline_actual,
    baseline_predictions
)

baseline_mae = baseline_metrics["MAE"]

print(
    f"Baseline GCN test MAE: {baseline_mae:.4f}"
)


# 7. Define the feature permutation strategy
calendar_features = {
    "time_idx",
    "month_sin",
    "month_cos"
}


def permute_graph_feature(
    graph,
    feature_index,
    feature_name,
    random_generator
):

    # Preserve the original graph and its target labels.
    permuted_graph = graph.clone()

    original_features = graph.x.reshape(
        encoder_length,
        n_parts,
        n_features
    )

    modified_features = (
        original_features.clone()
    )

    if feature_name in calendar_features:

        # Calendar features are shared across parts.
        # Permute historical month positions.
        permutation = torch.tensor(
            random_generator.permutation(
                encoder_length
            ),
            dtype=torch.long,
            device=original_features.device
        )

        modified_features[:, :, feature_index] = (
            original_features[
                permutation,
                :,
                feature_index
            ]
        )

        permutation_type = (
            "Historical month permutation"
        )

    else:

        # Exchange complete feature trajectories between parts.
        # Use the same part permutation at every historical month.
        permutation = torch.tensor(
            random_generator.permutation(
                n_parts
            ),
            dtype=torch.long,
            device=original_features.device
        )

        modified_features[:, :, feature_index] = (
            original_features[
                :,
                permutation,
                feature_index
            ]
        )

        permutation_type = (
            "Between-part trajectory permutation"
        )

    permuted_graph.x = (
        modified_features.reshape(
            encoder_length * n_parts,
            n_features
        )
    )

    return (
        permuted_graph,
        permutation_type
    )


# 8. Store the original features for integrity checking
original_feature_copies = [
    graph.x.clone()
    for graph in test_graphs
]


# 9. Repeat the permutation analysis for every feature
permutation_records = []

for feature_index, feature_name in enumerate(feature_names):

    print(
        f"\nEvaluating feature: {feature_name}"
    )

    for repeat in range(1, N_REPEATS + 1):

        permuted_graphs = []

        for graph in test_graphs:

            permuted_graph, permutation_type = (
                permute_graph_feature(
                    graph,
                    feature_index,
                    feature_name,
                    rng
                )
            )

            permuted_graphs.append(
                permuted_graph
            )

        actual_values, permuted_predictions = predict_graphs(
            gcn_model,
            permuted_graphs
        )

        # Permutations must not alter the test target values.
        np.testing.assert_array_equal(
            actual_values,
            baseline_actual
        )

        permuted_metrics = calculate_metrics(
            actual_values,
            permuted_predictions
        )

        delta_mae = (
            permuted_metrics["MAE"]
            - baseline_mae
        )

        permutation_records.append({
            "Feature": feature_name,
            "Repeat": repeat,
            "Permutation_Type": permutation_type,
            "Baseline_MAE": baseline_mae,
            "Permuted_MAE": permuted_metrics["MAE"],
            "Delta_MAE": delta_mae
        })


# 10. Verify that the original test graphs remain unchanged
for graph, original_features in zip(
    test_graphs,
    original_feature_copies
):

    assert torch.equal(
        graph.x,
        original_features
    )


# 11. Save the individual permutation results
permutation_df = pd.DataFrame(
    permutation_records
)

permutation_df.to_csv(
    output_folder / "GCN_permutation_repeats.csv",
    index=False
)


# 12. Summarize the repeated importance estimates
importance_df = (
    permutation_df
    .groupby("Feature")["Delta_MAE"]
    .agg(
        Mean_Delta_MAE="mean",
        Std_Delta_MAE="std",
        Min_Delta_MAE="min",
        Max_Delta_MAE="max",
        Repeats="count"
    )
    .reset_index()
)


# 13. Calculate an optional relative share of positive importance
# Keep signed importance estimates in the main results.
# The positive share is descriptive, not causal explained performance.
positive_importance = (
    importance_df["Mean_Delta_MAE"]
    .clip(lower=0)
)

positive_total = (
    positive_importance.sum()
)

if positive_total > 0:

    importance_df["Positive_Relative_Share_Pct"] = (
        100
        * positive_importance
        / positive_total
    )

else:

    importance_df["Positive_Relative_Share_Pct"] = 0.0

importance_df = importance_df.sort_values(
    "Mean_Delta_MAE",
    ascending=False
).reset_index(drop=True)

importance_df.to_csv(
    output_folder / "GCN_permutation_importance.csv",
    index=False
)

print("\nRepeated permutation importance:")
print(
    importance_df.round(4).to_string(index=False)
)


# 14. Plot signed importance estimates
plot_df = importance_df.sort_values(
    "Mean_Delta_MAE",
    ascending=True
)

bar_colors = [
    "#4472C4" if value >= 0 else "#ED7D31"
    for value in plot_df["Mean_Delta_MAE"]
]

fig, ax = plt.subplots(
    figsize=(9, 5)
)

ax.barh(
    plot_df["Feature"],
    plot_df["Mean_Delta_MAE"],
    xerr=plot_df["Std_Delta_MAE"],
    color=bar_colors,
    capsize=4
)

ax.axvline(
    0,
    color="black",
    linewidth=1
)

ax.set_xlabel(
    "Increase in test MAE after permutation"
)

ax.set_ylabel(
    "Input feature"
)

ax.set_title(
    "GCN permutation importance \n"
    "Error bars show one standard deviation across repeats"
)

ax.grid(
    axis="x",
    alpha=0.25
)

fig.tight_layout()

fig.savefig(
    output_folder / "GCN_permutation_importance.png",
    dpi=160
)

plt.show()
plt.close(fig)


# 15. Save the analysis settings
configuration = {
    "model": "GCN",
    "checkpoint": str(
        model_folder / "GCN_best_model.pt"
    ),
    "dataset": "test set",
    "permutation_seed": PERMUTATION_SEED,
    "repeats_per_feature": N_REPEATS,
    "importance_metric": (
        "Permuted MAE minus baseline MAE"
    ),
    "baseline_metrics": baseline_metrics,
    "part_feature_strategy": (
        "Exchange complete historical feature trajectories "
        "between parts."
    ),
    "calendar_feature_strategy": (
        "Permute historical month positions."
    ),
    "targets_unchanged": True,
    "original_graph_features_unchanged": True,
    "model_retrained": False,
    "interpretation": (
        "Signed importance estimates describe model reliance "
        "on this test set. Correlated features and potentially "
        "unrealistic feature combinations limit interpretation. "
        "Positive relative shares are not causal percentages "
        "of predictive performance."
    )
}

(
    output_folder / "GCN_permutation_configuration.json"
).write_text(
    json.dumps(configuration, indent=2),
    encoding="utf-8"
)

print(f"\nResults saved in: {output_folder}")